# MBTA Red Line: how late do trains run?

This notebook reads the PostgreSQL warehouse built by the pipeline and answers three questions:

1. How much data do we actually have, and when was it collected?
2. How far do trains drift from their first prediction?
3. Does that depend on the hour of the day or on the stop?

**Before you run it:** `docker compose up -d` (so `mbta_dw` is running) and a `.env` file in the project root with the `POSTGRES_*` values.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()  # finds the .env in the project root
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5433")),
    dbname=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

BOSTON = "America/New_York"   # service days and hours are read in Boston time
MIN_OBS = 5                   # ignore stop/hour groups with fewer observations than this
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. How much data do we have?

The pipeline only collects while the machine and Docker are running, so coverage has gaps. This is worth looking at before trusting any average below.

In [ ]:
coverage = pd.read_sql("""
    SELECT 'vehicle positions' AS table_name, COUNT(*) AS row_count,
           MIN(recorded_at) AS first_snapshot, MAX(recorded_at) AS last_snapshot
    FROM fact_vehicle_positions
    UNION ALL
    SELECT 'predictions', COUNT(*), MIN(recorded_at), MAX(recorded_at)
    FROM fact_predictions
""", conn)
coverage

In [ ]:
snap = pd.read_sql("SELECT DISTINCT recorded_at FROM fact_vehicle_positions", conn)
snap["recorded_at"] = pd.to_datetime(snap["recorded_at"], utc=True).dt.tz_convert(BOSTON)

by_hour = snap.groupby(snap["recorded_at"].dt.hour).size().reindex(range(24), fill_value=0)
by_day = snap.groupby(snap["recorded_at"].dt.date).size()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
by_hour.plot.bar(ax=axes[0], color="#4C78A8")
axes[0].set_title("Snapshots collected, by hour of day (Boston time)")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Snapshots")

by_day.plot.bar(ax=axes[1], color="#4C78A8")
axes[1].set_title("Snapshots collected, by day")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Snapshots")
plt.tight_layout()
plt.show()

## 2. Measuring delays

For each trip, stop and **Boston service day**, the delay is:

> first time a vehicle was seen `STOPPED_AT` that stop, minus the **first** arrival prediction recorded for it

Trip IDs are reused on different days, so the service day is part of the key. Without it, a prediction from one day would be paired with an arrival from another.

This is a proxy: the first prediction stands in for the official timetable, which this project doesn't ingest yet.

In [ ]:
delays = pd.read_sql("""
    WITH actual AS (
        SELECT trip_id, stop_id,
               (recorded_at AT TIME ZONE 'America/New_York')::date AS service_date,
               MIN(recorded_at) AS actual_arrival
        FROM fact_vehicle_positions
        WHERE current_status = 'STOPPED_AT'
        GROUP BY trip_id, stop_id, (recorded_at AT TIME ZONE 'America/New_York')::date
    ),
    first_pred AS (
        SELECT DISTINCT ON (trip_id, stop_id, (recorded_at AT TIME ZONE 'America/New_York')::date)
               trip_id, stop_id,
               (recorded_at AT TIME ZONE 'America/New_York')::date AS service_date,
               arrival_time AS first_predicted
        FROM fact_predictions
        WHERE arrival_time IS NOT NULL
        ORDER BY trip_id, stop_id, (recorded_at AT TIME ZONE 'America/New_York')::date, recorded_at
    )
    SELECT a.trip_id, a.stop_id, a.service_date, p.first_predicted, a.actual_arrival,
           EXTRACT(EPOCH FROM (a.actual_arrival - p.first_predicted)) AS delay_seconds,
           s.stop_name
    FROM actual a
    JOIN first_pred p USING (trip_id, stop_id, service_date)
    LEFT JOIN dim_stops s ON s.stop_id = a.stop_id
""", conn)

delays["actual_arrival"] = pd.to_datetime(delays["actual_arrival"], utc=True)
delays["hour"] = delays["actual_arrival"].dt.tz_convert(BOSTON).dt.hour
delays["delay_min"] = delays["delay_seconds"].astype(float) / 60
delays["stop_label"] = delays["stop_name"].fillna(delays["stop_id"])

print(f"{len(delays):,} trip/stop observations across {delays['service_date'].nunique()} service days")
delays["delay_min"].describe().round(2)

### Distribution of delays

A few very large values would flatten the chart, so the histogram is limited to a readable range. The count of values outside it is printed below, so nothing is hidden.

In [ ]:
LOW, HIGH = -5, 15
inside = delays[delays["delay_min"].between(LOW, HIGH)]
outside = len(delays) - len(inside)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(inside["delay_min"], bins=40, color="#4C78A8")
ax.axvline(0, color="#333", linewidth=1)
ax.set_title("Delay vs. first prediction (minutes)")
ax.set_xlabel("Minutes (negative = earlier than first predicted)")
ax.set_ylabel("Observations")
plt.tight_layout()
plt.show()

print(f"Shown: {len(inside):,} observations between {LOW} and {HIGH} min. Outside that range: {outside:,}.")

## 3. By hour of day

Only hours with at least `MIN_OBS` observations are shown. The number above each bar is how many observations it rests on, so thin bars are easy to spot.

In [ ]:
by_hour_delay = (
    delays.groupby("hour")["delay_min"]
    .agg(avg_delay="mean", n="count")
    .query("n >= @MIN_OBS")
)

if by_hour_delay.empty:
    print(f"No hour has {MIN_OBS}+ observations yet. Let the pipeline collect more data.")
else:
    fig, ax = plt.subplots(figsize=(10, 3.8))
    bars = ax.bar(by_hour_delay.index.astype(str), by_hour_delay["avg_delay"], color="#4C78A8")
    for bar, n in zip(bars, by_hour_delay["n"]):
        ax.annotate(f"n={n}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha="center", va="bottom", fontsize=8)
    ax.axhline(0, color="#333", linewidth=1)
    ax.set_title("Average delay by hour of day (Boston time)")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Average delay (minutes)")
    plt.tight_layout()
    plt.show()

## 4. By stop

Stops are shown by name when the warehouse has one. Platform-level stop IDs created on the fly during loading have no name yet, so they appear as numeric IDs.

In [ ]:
by_stop = (
    delays.groupby("stop_label")["delay_min"]
    .agg(avg_delay="mean", n="count")
    .query("n >= @MIN_OBS")
    .sort_values("avg_delay", ascending=False)
    .head(10)
)

if by_stop.empty:
    print(f"No stop has {MIN_OBS}+ observations yet.")
else:
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.barh(by_stop.index[::-1], by_stop["avg_delay"][::-1], color="#4C78A8")
    for i, (avg, n) in enumerate(zip(by_stop["avg_delay"][::-1], by_stop["n"][::-1])):
        ax.annotate(f" n={n}", (avg, i), va="center", fontsize=8)
    ax.set_title(f"Stops with the highest average delay (at least {MIN_OBS} observations)")
    ax.set_xlabel("Average delay (minutes)")
    plt.tight_layout()
    plt.show()

## 5. The Spark job's output

`spark_jobs/delay_analysis.py` computes the same kind of summary with PySpark and writes it to `fact_delay_summary`. If you haven't run it yet, this cell says so.

In [ ]:
try:
    spark_summary = pd.read_sql(
        "SELECT * FROM fact_delay_summary ORDER BY num_observations DESC LIMIT 15", conn
    )
    display(spark_summary)
except Exception as exc:
    conn.rollback()
    print("fact_delay_summary not found. Run: python spark_jobs/delay_analysis.py")

## Reading these charts honestly

- **Sampling interval.** The pipeline takes a snapshot every 2 minutes, so an arrival can be noticed up to 2 minutes late, and a short stop can be missed. Delays here can be overstated by about that much. This effect hasn't been measured yet.
- **"Expected" is a proxy.** The first prediction is not the official schedule.
- **Coverage gaps.** Data exists only for the periods the pipeline was running. Check section 1 before comparing hours or days.
- **Small groups.** Averages over a handful of observations are noisy, which is why every chart shows `n` or filters on `MIN_OBS`.
- **Scope.** Red Line only. Shuttle-bus trips are excluded.